# Missingness After Merging

**DS4DH Practice Pack · Module 03 — Integrating Multiple Data Sources**

*Technique:* Structural vs join-induced missingness, and conservation checks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/03b_missingness_after_merge.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

After a merge there are two kinds of empty cell, and they call for opposite
responses:

- **Structural** — the value was never collected or was suppressed at source.
  Nothing you do in code recovers it. It is part of what the data is.
- **Join-induced** — the value exists, but your key failed to find it. This is a
  bug, and it is fixable.

Treating a bug as a fact of the world is how an analysis quietly loses a third of
its sample.

In [ ]:
csd = df.dropna(subset=['csd_code'])
cols = ['Total', 'Owner', 'Renter', 'tot_income', 'own_income', 'rent_income']

print('Missingness in the shipped merged file:')
print()
print(f'{"column":<16}{"missing":>9}{"pct":>8}')
print('-' * 33)
for c in cols:
    print(f'{c:<16}{csd[c].isna().sum():>9}{csd[c].isna().mean():>8.1%}')

## Telling the two apart

The test is whether the missingness follows a pattern that the *source* would
produce, or one that a *key* would produce.

Suppression is population-dependent: small places lose cells. A failed join is
usually all-or-nothing for whole blocks of rows, and is independent of size.

In [ ]:
sized = csd[csd['immigrant_status'] == 'Total Immigrant Status'].dropna(subset=['tot_pop'])

for c in ['Renter', 'rent_income']:
    have = sized[sized[c].notna()]['tot_pop'].median()
    gone = sized[sized[c].isna()]['tot_pop'].median()
    print(f'{c:<14} median pop where present: {have:>10,.0f}')
    print(f'{"":<14} median pop where absent:  {gone:>10,.0f}')
    print(f'{"":<14} ratio: {have / gone:>5.1f}x  -> {"structural (size-linked)" if have > gone * 2 else "check the join"}')
    print()

In [ ]:
# A join failure looks different: it clusters by source, not by size.
print('Missing Renter STIR by immigrant status:')
print()
for grp in ['Immigrant', 'Non-immigrants', 'Total Immigrant Status']:
    s = csd[csd['immigrant_status'] == grp]
    print(f'  {grp:<24}{s["Renter"].isna().sum():>4} / {len(s):<4}  ({s["Renter"].isna().mean():.1%})')
print()
print('All three groups lose a similar share. A broken join would typically')
print('wipe out one group entirely while leaving the others intact.')

### 🔧 Your turn 1

Run the same by-group breakdown for `rent_income` instead of `Renter`.

Is the loss even across groups, or concentrated? What would you conclude?

## Conservation checks

A conservation check states something that must remain true after a merge, and
fails loudly when it does not. Three that apply here.

In [ ]:
checks = []

# 1. Row count is preserved per geography.
per_place = csd.groupby('csd_code').size()
checks.append(('every CSD has exactly 3 rows', bool((per_place == 3).all())))

# 2. No key appears that was not in the source.
checks.append(('no orphan join_keys', bool(csd['join_key'].notna().all())))

# 3. Renter STIR, where present, is a plausible percentage.
r = csd['Renter'].dropna()
checks.append(('Renter STIR within 0-100', bool(((r >= 0) & (r <= 100)).all())))

for name, ok in checks:
    print(f'  {"PASS" if ok else "FAIL"}  {name}')

In [ ]:
# Totals should reconcile: the 'Total Immigrant Status' row is not the mean of
# the other two, because the groups differ in size. Check that it is bracketed
# by them, which it must be for any weighted average.
wide = csd.pivot_table(index='csd_code', columns='immigrant_status',
                       values='Renter', aggfunc='first').dropna()
lo = wide[['Immigrant', 'Non-immigrants']].min(axis=1)
hi = wide[['Immigrant', 'Non-immigrants']].max(axis=1)
within = ((wide['Total Immigrant Status'] >= lo - 0.05)
          & (wide['Total Immigrant Status'] <= hi + 0.05))

print(f'CSDs with all three values: {len(wide)}')
print(f'  total bracketed by the two groups: {within.sum()} ({within.mean():.1%})')
print(f'  outside the bracket:               {(~within).sum()}')
print()
if (~within).any():
    print('Exceptions — worth inspecting rather than ignoring:')
    print(wide[~within].head(5).to_string())

### 🔧 Your turn 2

Look at any rows that fall outside the bracket.

Rounding to one decimal place can push a value marginally outside. Widen the
tolerance from `0.05` to `0.5` and re-run — do they all resolve? If some do not,
that is a genuine data question worth raising with the source.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** `rent_income` is missing at a similar rate across all three
groups and is strongly size-linked — the median population where it is present is
several times the median where it is absent. Both signals point to structural
suppression, not a join failure. The right response is to document the reduced
sample and move on, not to impute.

**Your turn 2.** Most exceptions resolve at a 0.5 tolerance: the published values
are rounded to one decimal, and a weighted average of two rounded numbers can sit
a fraction outside the bracket of the rounded inputs. Any that remain after
widening are worth flagging — they usually indicate that the "total" row was
computed on a slightly different household universe than the two component rows,
which is common in census tables and worth a footnote.

The habit that matters: a conservation check that fails is not a nuisance to be
tuned away. It is the dataset telling you something about how it was built.

</details>

## Where this stops

Your merged table is now trustworthy in structure and honest about its gaps.
Module 04 starts asking whether the differences in it are real.